# Recomendaciones top-20 (solo predicción, sin reentrenar)

Reusa el modelo ya entrenado en `modelo_lightgbm.ipynb` (guardado en
`datos/modelo_lightgbm_artifacts.joblib`: el `LightGBMRegressor` final, los mapas de target encoding de
autor/editorial, y las categorías vistas en entrenamiento) para volver a generar el top-20 por lector
sin repetir la búsqueda bayesiana ni el entrenamiento.

**Cambio respecto a la corrida anterior**: el catálogo de candidatos ya no exige un mínimo de 5
calificaciones — solo se excluyen los libros que no tuvieron **ninguna** valoración en `interacciones`
(`MIN_RATINGS_RECOMENDABLE = 1`). El resto del pipeline (perfil de género actual por lector, exclusión
de libros ya leídos, predicción en lotes, validaciones) es igual al de `modelo_lightgbm.ipynb`.

In [1]:
import re
import sqlite3
import time
import unicodedata
import warnings
from datetime import date

import joblib
import numpy as np
import pandas as pd
from rapidfuzz import fuzz

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

DB_PATH = "datos/data.db"
EJEMPLO_CSV = "datos/ejemplo.csv"
ARTIFACTS_PATH = "datos/modelo_lightgbm_artifacts.joblib"
FECHA_HOY = date.today().isoformat()
OUTPUT_CSV = f"datos/recomendaciones_top20_lightgbm_{FECHA_HOY}.csv"
RANDOM_STATE = 42
MIN_RATINGS_RECOMENDABLE = 1  # solo se excluyen libros con CERO valoraciones en interacciones


## 1. Cargar artefactos del modelo entrenado

In [2]:
artefactos = joblib.load(ARTIFACTS_PATH)
final_model = artefactos["final_model"]
encodings_full = artefactos["encodings_full"]
cat_categories = artefactos["cat_categories"]
FEATURE_COLS = artefactos["FEATURE_COLS"]
CAT_COLS = artefactos["CAT_COLS"]
ENC_COLS = artefactos["ENC_COLS"]

print(f"modelo: {final_model.n_estimators} arboles")
print(f"features: {len(FEATURE_COLS)}  categoricas nativas: {CAT_COLS}  target-encoded: {ENC_COLS}")


def aplicar_encoding(cat_series, enc_map, media_global):
    return cat_series.map(enc_map).fillna(media_global)

modelo: 1173 arboles
features: 220  categoricas nativas: ['genero_libro', 'genero_lector', 'pais_lector']  target-encoded: ['autor_libro', 'editorial_libro']


## 2. Perfil de género actual por lector

Mismo pipeline que `dataset_features_genero.ipynb` / `modelo_lightgbm.ipynb`: fuzzy matching de género +
changelog acumulado por lector×género vía SQL, y nos quedamos con el último estado de cada lector (todo
su historial hasta hoy, sin corte de fecha — acá no hay fuga posible porque predecimos sobre libros que
el lector no leyó).

In [3]:
def agrupar_por_similitud(items, threshold, scorer=fuzz.ratio):
    # une items en clusters (union-find) segun similitud; devuelve tambien los pares que dispararon cada union
    padre = {it: it for it in items}

    def find(x):
        while padre[x] != x:
            padre[x] = padre[padre[x]]
            x = padre[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            padre[ra] = rb

    pares = []
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            score = scorer(items[i], items[j])
            if score >= threshold:
                pares.append((items[i], items[j], score))
                union(items[i], items[j])

    clusters = {}
    for it in items:
        clusters.setdefault(find(it), []).append(it)
    return clusters, pares


def slugify(genero_norm: str) -> str:
    s = unicodedata.normalize("NFKD", genero_norm).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")

In [4]:
conn = sqlite3.connect(DB_PATH)

q_conteo_genero = """
SELECT LOWER(TRIM(l.genero)) AS g, COUNT(*) AS n
FROM interacciones i JOIN libros l ON l.id_libro = i.id_libro
WHERE l.genero IS NOT NULL AND TRIM(l.genero) <> ''
GROUP BY g
"""
conteo_genero = pd.read_sql_query(q_conteo_genero, conn).set_index("g")["n"].to_dict()
clusters_genero, _ = agrupar_por_similitud(sorted(conteo_genero.keys()), 80)

genero_canon = {"desconocido": "desconocido"}
for miembros in clusters_genero.values():
    canonico = max(miembros, key=lambda m: conteo_genero.get(m, 0))
    for m in miembros:
        genero_canon[m] = canonico
print(f"genero_canon: {len(genero_canon)} generos originales -> {len(set(genero_canon.values()))} canonicos")

genero_canon: 55 generos originales -> 53 canonicos


In [5]:
conn.execute("CREATE TEMP TABLE genero_canon (genero_norm TEXT PRIMARY KEY, genero_canonico TEXT NOT NULL)")
conn.executemany("INSERT INTO genero_canon VALUES (?, ?)", list(genero_canon.items()))

query_changelog = """
WITH base AS (
    SELECT
        i.id_lector,
        COALESCE(gc.genero_canonico, 'desconocido') AS genero_norm,
        substr(i.fecha, 7, 4) || '-' || substr(i.fecha, 4, 2) || '-' || substr(i.fecha, 1, 2) AS fecha_iso,
        i.rating AS rating
    FROM interacciones i
    JOIN libros l    ON l.id_libro = i.id_libro
    JOIN lectores r  ON r.id_lector = i.id_lector
    LEFT JOIN genero_canon gc ON gc.genero_norm = LOWER(TRIM(l.genero))
    WHERE i.fecha GLOB '[0-9][0-9]-[0-9][0-9]-[0-9][0-9][0-9][0-9]'
),
por_dia AS (
    SELECT
        id_lector, genero_norm, fecha_iso,
        COUNT(*) AS n_dia, SUM(rating) AS suma_dia, MIN(rating) AS min_dia, MAX(rating) AS max_dia
    FROM base
    GROUP BY id_lector, genero_norm, fecha_iso
)
SELECT
    id_lector, genero_norm, fecha_iso,
    SUM(n_dia)    OVER w AS n_acum,
    SUM(suma_dia) OVER w AS suma_acum,
    MIN(min_dia)  OVER w AS min_acum,
    MAX(max_dia)  OVER w AS max_acum
FROM por_dia
WINDOW w AS (
    PARTITION BY id_lector, genero_norm
    ORDER BY fecha_iso
    RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
"""
t0 = time.time()
changelog = pd.read_sql_query(query_changelog, conn)
changelog["fecha_iso"] = pd.to_datetime(changelog["fecha_iso"])
print(f"changelog: {changelog.shape} en {time.time() - t0:.1f}s")

changelog: (246671, 7) en 1.6s


In [6]:
genero_slug = {g: slugify(g) for g in changelog["genero_norm"].unique()}
assert len(set(genero_slug.values())) == len(genero_slug)

wide_hist = changelog.pivot(
    index=["id_lector", "fecha_iso"], columns="genero_norm",
    values=["n_acum", "suma_acum", "min_acum", "max_acum"],
)
wide_hist.columns = [
    f"genero_hist__{metric.replace('_acum', '')}__{genero_slug[genero]}"
    for metric, genero in wide_hist.columns
]
wide_hist = wide_hist.reset_index().sort_values("fecha_iso")

n_cols_hist = [c for c in wide_hist.columns if c.startswith("genero_hist__n__")]
otras_cols = [c for c in wide_hist.columns if c.startswith("genero_hist__") and c not in n_cols_hist]
wide_hist[n_cols_hist] = wide_hist.groupby("id_lector")[n_cols_hist].ffill()
wide_hist[otras_cols] = wide_hist.groupby("id_lector")[otras_cols].ffill()

# perfil actual = ultimo estado de cada lector (todo su historial, sin cortar por fecha)
perfil_actual = wide_hist.sort_values("fecha_iso").groupby("id_lector").tail(1).drop(columns=["fecha_iso"])

suma_cols = [c for c in perfil_actual.columns if c.startswith("genero_hist__suma__")]
avg_nuevas = {}
for suma_c in suma_cols:
    tag = suma_c[len("genero_hist__suma__"):]
    n_c = f"genero_hist__n__{tag}"
    avg_nuevas[f"genero_hist__avg__{tag}"] = perfil_actual[suma_c] / perfil_actual[n_c].replace(0, np.nan)
perfil_actual = pd.concat([perfil_actual.drop(columns=suma_cols), pd.DataFrame(avg_nuevas, index=perfil_actual.index)], axis=1)
perfil_actual[n_cols_hist] = perfil_actual[n_cols_hist].fillna(0).astype("int32")

min_max_cols = [c for c in perfil_actual.columns if "__min__" in c or "__max__" in c]
perfil_actual[min_max_cols] = perfil_actual[min_max_cols].astype("float32")

print(f"perfil_actual: {perfil_actual.shape} (una fila por lector con historial)")
hist_feature_cols = [c for c in FEATURE_COLS if c.startswith("genero_hist__")]
faltan = set(hist_feature_cols) - set(perfil_actual.columns)
sobran = set(perfil_actual.columns) - set(hist_feature_cols) - {"id_lector"}
assert not faltan and not sobran, f"desalineacion de columnas de historial: faltan={faltan} sobran={sobran}"
print("OK: las columnas de historial coinciden exactamente con las de entrenamiento.")

perfil_actual: (10667, 213) (una fila por lector con historial)
OK: las columnas de historial coinciden exactamente con las de entrenamiento.


## 3. Catálogo de candidatos (>= 1 calificación válida)

In [7]:
q_candidatos = """
SELECT
    l.id_libro,
    l.autor        AS autor_libro,
    l.genero        AS genero_libro_raw,
    l.editorial     AS editorial_libro,
    l.anio_edicion  AS anio_edicion_libro,
    conteo.n_ratings  AS n_ratings,
    conteo.avg_rating AS avg_rating
FROM libros l
JOIN (
    SELECT i.id_libro, COUNT(*) AS n_ratings, AVG(i.rating) AS avg_rating
    FROM interacciones i
    JOIN lectores r ON r.id_lector = i.id_lector
    WHERE i.fecha GLOB '[0-9][0-9]-[0-9][0-9]-[0-9][0-9][0-9][0-9]'
    GROUP BY i.id_libro
) conteo ON conteo.id_libro = l.id_libro
WHERE conteo.n_ratings >= ?
"""
candidatos = pd.read_sql_query(q_candidatos, conn, params=(MIN_RATINGS_RECOMENDABLE,))

candidatos["genero_libro"] = (
    candidatos["genero_libro_raw"].str.strip().str.lower().map(genero_canon).fillna("desconocido")
)
anio_num = pd.to_numeric(candidatos["anio_edicion_libro"], errors="coerce")
candidatos["anio_edicion_libro"] = anio_num.where(anio_num.between(1400, 2026))

# libro_hist__n / libro_hist__avg = estado actual del historial del libro (todo su historial hasta
# hoy, sin corte de fecha -- mismo criterio que el perfil de género actual: acá no hay fuga posible
# porque predecimos sobre libros que el lector no leyó)
candidatos["libro_hist__n"] = candidatos["n_ratings"].astype("int32")
candidatos["libro_hist__avg"] = candidatos["avg_rating"].astype("float32")
candidatos = candidatos.drop(columns=["genero_libro_raw", "n_ratings", "avg_rating"])

for c in ENC_COLS:
    enc_map, media_global = encodings_full[c]
    candidatos[f"{c}_enc"] = aplicar_encoding(candidatos[c], enc_map, media_global)
candidatos = candidatos.drop(columns=ENC_COLS)

candidatos["genero_libro"] = candidatos["genero_libro"].astype(pd.CategoricalDtype(categories=cat_categories["genero_libro"]))

libro_hist_feature_cols = [c for c in FEATURE_COLS if c.startswith("libro_hist__")]
faltan_libro_hist = set(libro_hist_feature_cols) - set(candidatos.columns)
assert not faltan_libro_hist, f"faltan columnas de historial de libro en candidatos: {faltan_libro_hist}"

print(f"libros candidatos (>= {MIN_RATINGS_RECOMENDABLE} calificacion/es): {len(candidatos):,}")
print("nulos en columnas de candidatos:")
print(candidatos.isna().sum()[candidatos.isna().sum() > 0])

libros candidatos (>= 1 calificacion/es): 48,063
nulos en columnas de candidatos:
anio_edicion_libro    83
dtype: int64


## 4. Lectores objetivo (`ejemplo.csv`) y libros ya leídos

In [8]:
ejemplo = pd.read_csv(EJEMPLO_CSV)
lectores_objetivo = sorted(ejemplo["id_lector"].unique())
print(f"lectores objetivo: {len(lectores_objetivo)}")

lectores_tabla = pd.read_sql_query("SELECT id_lector, genero AS genero_lector, vive_en FROM lectores", conn)
lectores_tabla["genero_lector"] = lectores_tabla["genero_lector"].replace({"-": np.nan, "": np.nan})
lectores_tabla["pais_lector"] = (
    lectores_tabla["vive_en"].str.rsplit("-", n=1).str[-1].str.strip().str.lower()
    .replace({"": np.nan, "¿?": np.nan})
)
lectores_tabla = lectores_tabla.drop(columns=["vive_en"])
lectores_tabla["genero_lector"] = lectores_tabla["genero_lector"].astype(pd.CategoricalDtype(categories=cat_categories["genero_lector"]))
lectores_tabla["pais_lector"] = lectores_tabla["pais_lector"].astype(pd.CategoricalDtype(categories=cat_categories["pais_lector"]))

leidos = pd.read_sql_query("SELECT DISTINCT id_lector, id_libro FROM interacciones", conn)
conn.close()

perfil_lectores_obj = (
    pd.DataFrame({"id_lector": lectores_objetivo})
    .merge(lectores_tabla, on="id_lector", how="left")
    .merge(perfil_actual, on="id_lector", how="left")
)
n_cols_obj = [c for c in perfil_lectores_obj.columns if c.startswith("genero_hist__n__")]
perfil_lectores_obj[n_cols_obj] = perfil_lectores_obj[n_cols_obj].fillna(0)
perfil_lectores_obj = perfil_lectores_obj.copy()

sin_historial = perfil_lectores_obj["genero_hist__n__narrativa"].isna().sum()
print(f"lectores objetivo sin ningun historial previo (perfil vacio, cold-start): {sin_historial}")

lectores objetivo: 832


lectores objetivo sin ningun historial previo (perfil vacio, cold-start): 0


## 5. Predicción en batches y top-20 por lector

In [9]:
BATCH_SIZE = 30
n_batches = int(np.ceil(len(perfil_lectores_obj) / BATCH_SIZE))
print(f"lectores: {len(perfil_lectores_obj)}  candidatos: {len(candidatos):,}  lotes: {n_batches}")

resultados = []
t_total = time.time()
for bi in range(n_batches):
    lote = perfil_lectores_obj.iloc[bi * BATCH_SIZE:(bi + 1) * BATCH_SIZE]

    cross = lote.merge(candidatos, how="cross")
    cross = cross.merge(leidos.assign(_leido=1), on=["id_lector", "id_libro"], how="left")
    cross = cross[cross["_leido"].isna()].drop(columns=["_leido"])

    cross["pred_rating"] = final_model.predict(cross[FEATURE_COLS])

    top20 = (
        cross.sort_values(["id_lector", "pred_rating"], ascending=[True, False])
        .groupby("id_lector", sort=False)
        .head(20)[["id_lector", "id_libro", "pred_rating"]]
    )
    resultados.append(top20)

    if (bi + 1) % 5 == 0 or bi == n_batches - 1:
        print(f"  lote {bi + 1}/{n_batches} ({time.time() - t_total:.0f}s acumulados)")

recomendaciones = pd.concat(resultados, ignore_index=True)
print(f"\ntotal: {recomendaciones.shape} en {time.time() - t_total:.0f}s")

lectores: 832  candidatos: 48,063  lotes: 28


  lote 5/28 (38s acumulados)


  lote 10/28 (75s acumulados)


  lote 15/28 (114s acumulados)


  lote 20/28 (152s acumulados)


  lote 25/28 (190s acumulados)


  lote 28/28 (213s acumulados)

total: (16640, 3) en 213s


## 6. Validaciones de sanidad

In [10]:
filas_por_lector = recomendaciones.groupby("id_lector").size()
print("lectores objetivo cubiertos:", filas_por_lector.index.isin(lectores_objetivo).all() and len(filas_por_lector) == len(lectores_objetivo))
print("distribucion de filas por lector:")
print(filas_por_lector.value_counts())

incompletos = filas_por_lector[filas_por_lector < 20]
if len(incompletos):
    print(f"\n{len(incompletos)} lectores con menos de 20 recomendaciones (menos de 20 candidatos sin leer disponibles):")
    print(incompletos)

dup = recomendaciones.duplicated(subset=["id_lector", "id_libro"]).sum()
print(f"\npares (lector, libro) duplicados: {dup}")

ya_leidos_set = set(map(tuple, leidos[["id_lector", "id_libro"]].itertuples(index=False, name=None)))
recomendados_set = set(map(tuple, recomendaciones[["id_lector", "id_libro"]].itertuples(index=False, name=None)))
interseccion = recomendados_set & ya_leidos_set
print(f"recomendaciones que en realidad ya estaban leidas (deberia ser 0): {len(interseccion)}")

assert dup == 0, "hay pares (lector, libro) duplicados"
assert len(interseccion) == 0, "se recomendo un libro ya leido"
print("\nOK: sin duplicados, sin libros ya leidos.")

lectores objetivo cubiertos: True
distribucion de filas por lector:
20    832
Name: count, dtype: int64

pares (lector, libro) duplicados: 0
recomendaciones que en realidad ya estaban leidas (deberia ser 0): 0

OK: sin duplicados, sin libros ya leidos.


## 7. CSV final (misma estructura que `ejemplo.csv`, con la fecha de hoy en el nombre)

In [11]:
salida = (
    recomendaciones.sort_values(["id_lector", "pred_rating"], ascending=[True, False])
    [["id_lector", "id_libro"]]
    .reset_index(drop=True)
)

print("columnas:", salida.columns.tolist(), "== ejemplo.csv:", salida.columns.tolist() == ejemplo.columns.tolist())
print("filas:", len(salida), " (ejemplo.csv tiene", len(ejemplo), ")")
print("lectores unicos:", salida["id_lector"].nunique(), " (ejemplo.csv tiene", ejemplo["id_lector"].nunique(), ")")

salida.to_csv(OUTPUT_CSV, index=False)
print(f"\nguardado en {OUTPUT_CSV}")
salida.head(10)

columnas: ['id_lector', 'id_libro'] == ejemplo.csv: True
filas: 16640  (ejemplo.csv tiene 16640 )
lectores unicos: 832  (ejemplo.csv tiene 832 )



guardado en datos/recomendaciones_top20_lightgbm_2026-08-18.csv


,id_lector,id_libro
0,05-03-1970,la-realidad-insuficiente-una-critica-precisa-a...
1,05-03-1970,con-la-luna-en-el-bolsillo
2,05-03-1970,nietzsche-4
3,05-03-1970,railes-y-maletas
4,05-03-1970,el-monstruo-del-monoculo-y-otras-bestias-secre...
5,05-03-1970,corea-en-100-palabras
6,05-03-1970,la-radio-ante-el-microfono-voz-erotismo-y-soci...
7,05-03-1970,el-verdadero-tercer-hombre
8,05-03-1970,el-estornino-de-mozart
9,05-03-1970,la-ciencia-de-contar-historias-por-que-las-his...


## 8. Resumen y ejemplo

In [12]:
conn = sqlite3.connect(DB_PATH)
libros_info = pd.read_sql_query("SELECT id_libro, titulo, autor, genero FROM libros", conn)
conn.close()

muestra = recomendaciones["id_lector"].drop_duplicates().sample(2, random_state=RANDOM_STATE).tolist()
for lid in muestra:
    top5 = recomendaciones[recomendaciones["id_lector"] == lid].sort_values("pred_rating", ascending=False).head(5)
    top5 = top5.merge(libros_info, on="id_libro", how="left")
    print(f"\n=== top 5 para {lid} ===")
    print(top5[["id_libro", "titulo", "autor", "genero", "pred_rating"]].to_string(index=False))


=== top 5 para omallorqui ===
                                                                 id_libro                                                                    titulo                                                   autor                 genero  pred_rating
                                                          dolor-y-memoria                                                           DOLOR Y MEMORIA CUADRADO, AURORA; RODRÍGUEZ, DANIEL y TAPIAS, FRANCISCO Cómics, Novela Gráfica     9.146262
            francisco-de-goya-henri-rousseau-van-gogh-del-lienzo-al-comic          FRANCISCO DE GOYA, HENRI ROUSSEAU, VAN GOGH. DEL LIENZO AL CÓMIC                                    MONTES, INGE EGUILUZ Cómics, Novela Gráfica     9.146262
historias-extraordinarias-de-las-matematicas-y-de-la-informatica-en-comic HISTORIAS EXTRAORDINARIAS DE LAS MATEMÁTICAS Y DE LA INFORMÁTICA EN CÓMIC                                            FINTZ, NESIM Cómics, Novela Gráfica     9.142379
         

**Resumen**: mismo modelo que `modelo_lightgbm.ipynb` (cargado desde
`datos/modelo_lightgbm_artifacts.joblib`, sin reentrenar), pero el catálogo de candidatos ahora incluye
todos los libros con al menos una calificación válida en `interacciones` (antes se exigían >= 5). El
resultado se guarda en un CSV con la fecha del día para distinguirlo de la corrida anterior
(`datos/recomendaciones_top20_lightgbm.csv`).